# Pipeline: Resident Education Drop Risk (Next Month)

## Problem framing
**Who cares**: Case management + education coordinator.

**Business question**: Which residents are at risk of a meaningful drop in **education progress next month**, so staff can intervene early.

**Predictive goal**: predict `edu_delta_next` (regression on the month-over-month change in progress_percent). A negative prediction = expected drop.

**Label definition** (default):
- `edu_drop_next_month = 1` if next month’s `progress_percent` is at least **10 points lower** than this month.

**Explanatory goal**: identify which factors (incidents, low attendance, low sessions/visits) are most associated.

## Output
- `predictions_resident_edu_drop_next_month.csv`

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 200)

HERE = Path.cwd()
if (HERE / "education_records.csv").exists():
    DATA_DIR = HERE
elif (HERE.parent / "education_records.csv").exists():
    DATA_DIR = HERE.parent
else:
    raise FileNotFoundError("Could not locate education_records.csv from current working directory")

residents = pd.read_csv(DATA_DIR / "residents.csv")
edu = pd.read_csv(DATA_DIR / "education_records.csv")
incidents = pd.read_csv(DATA_DIR / "incident_reports.csv")
visits = pd.read_csv(DATA_DIR / "home_visitations.csv")
sessions = pd.read_csv(DATA_DIR / "process_recordings.csv")

edu["record_date"] = pd.to_datetime(edu["record_date"], errors="coerce")
incidents["incident_date"] = pd.to_datetime(incidents["incident_date"], errors="coerce")
visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")
sessions["session_date"] = pd.to_datetime(sessions["session_date"], errors="coerce")

residents.shape, edu.shape

((60, 49), (534, 10))

In [2]:
# Build resident-month table for education

edu_rm = edu.copy()
edu_rm["month_start"] = edu_rm["record_date"].dt.to_period("M").dt.to_timestamp()

# Aggregate in case there are multiple rows per month
edu_rm = edu_rm.groupby(["resident_id", "month_start"]).agg(
    edu_progress=("progress_percent", "mean"),
    edu_attendance=("attendance_rate", "mean"),
    enrollment_status=("enrollment_status", lambda s: s.dropna().astype(str).mode().iloc[0] if s.notna().any() else None),
    education_level=("education_level", lambda s: s.dropna().astype(str).mode().iloc[0] if s.notna().any() else None),
    completion_status=("completion_status", lambda s: s.dropna().astype(str).mode().iloc[0] if s.notna().any() else None),
).reset_index()

# Add incident/session/visit counts in the same month
incidents["month_start"] = incidents["incident_date"].dt.to_period("M").dt.to_timestamp()
inc_rm = incidents.groupby(["resident_id", "month_start"]).agg(
    incidents_m=("incident_id", "count"),
    high_sev_m=("severity", lambda s: (s.astype(str).str.lower() == "high").sum()),
).reset_index()

sessions["month_start"] = sessions["session_date"].dt.to_period("M").dt.to_timestamp()
sess_rm = sessions.groupby(["resident_id", "month_start"]).agg(
    sessions_m=("recording_id", "count"),
    session_minutes_m=("session_duration_minutes", "sum"),
    concerns_m=("concerns_flagged", lambda s: s.fillna(False).astype(bool).sum()),
).reset_index()

visits["month_start"] = visits["visit_date"].dt.to_period("M").dt.to_timestamp()
vis_rm = visits.groupby(["resident_id", "month_start"]).agg(
    visits_m=("visitation_id", "count"),
    safety_concerns_m=("safety_concerns_noted", lambda s: s.fillna(False).astype(bool).sum()),
).reset_index()

rm = edu_rm.merge(inc_rm, on=["resident_id", "month_start"], how="left") \
          .merge(sess_rm, on=["resident_id", "month_start"], how="left") \
          .merge(vis_rm, on=["resident_id", "month_start"], how="left")

for c in ["incidents_m", "high_sev_m", "sessions_m", "session_minutes_m", "concerns_m", "visits_m", "safety_concerns_m"]:
    rm[c] = pd.to_numeric(rm[c], errors="coerce").fillna(0)

# Add static resident context
static_cols = [
    "safehouse_id",
    "case_status",
    "case_category",
    "initial_risk_level",
    "current_risk_level",
    "referral_source",
    "has_special_needs",
    "is_pwd",
    "sex",
    "sub_cat_orphaned",
    "sub_cat_trafficked",
    "sub_cat_child_labor",
    "sub_cat_physical_abuse",
    "sub_cat_sexual_abuse",
    "sub_cat_osaec",
    "sub_cat_at_risk",
    "family_is_4ps",
    "family_solo_parent",
    "family_indigenous",
    "family_informal_settler",
    "age_upon_admission",
    "present_age",
]
static_cols = [c for c in static_cols if c in residents.columns]
rm = rm.merge(residents[["resident_id"] + static_cols], on="resident_id", how="left")

rm = rm.sort_values(["resident_id", "month_start"]).reset_index(drop=True)
rm.head()

,resident_id,month_start,edu_progress,edu_attendance,enrollment_status,education_level,completion_status,incidents_m,high_sev_m,sessions_m,session_minutes_m,concerns_m,visits_m,safety_concerns_m,safehouse_id,case_status,case_category,initial_risk_level,current_risk_level,referral_source,has_special_needs,is_pwd,sex,sub_cat_orphaned,sub_cat_trafficked,sub_cat_child_labor,sub_cat_physical_abuse,sub_cat_sexual_abuse,sub_cat_osaec,sub_cat_at_risk,family_is_4ps,family_solo_parent,family_indigenous,family_informal_settler,age_upon_admission,present_age
0,1,2023-10-01,37.7,0.966,Enrolled,Vocational,NotStarted,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months
1,1,2023-11-01,33.0,0.693,Enrolled,Secondary,InProgress,0.0,0.0,3.0,222.0,2.0,2.0,1.0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months
2,1,2023-12-01,54.0,0.744,Enrolled,Vocational,InProgress,0.0,0.0,6.0,452.0,0.0,5.0,0.0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months
3,1,2024-01-01,51.2,0.681,Enrolled,Primary,InProgress,0.0,0.0,3.0,182.0,1.0,2.0,2.0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months
4,1,2024-02-01,44.2,0.721,Enrolled,Vocational,InProgress,1.0,0.0,3.0,133.0,0.0,1.0,0.0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months


In [3]:
# Target: next-month education progress delta (regression)

rm["edu_progress_next"] = rm.groupby("resident_id")["edu_progress"].shift(-1)
rm["edu_delta_next"] = rm["edu_progress_next"] - rm["edu_progress"]

model_df = rm.dropna(subset=["edu_progress_next"]).copy()

TARGET = "edu_delta_next"
exclude = {"resident_id", "month_start", "edu_progress_next", "edu_delta_next", "edu_drop_next_month"}
feature_cols = [c for c in model_df.columns if c not in exclude]

X_all = model_df[feature_cols].copy()
y_all = model_df[TARGET].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in numeric_cols]

for c in cat_cols:
    X_train[c] = X_train[c].astype("object")
    X_test[c] = X_test[c].astype("object")

pre = ColumnTransformer(
    [
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
}

best_name, best_score, best_pipe = None, -1e9, None
for name, model in models.items():
    p = Pipeline([("pre", pre), ("model", model)])
    cv = cross_val_score(p, X_train, y_train, cv=5, scoring="r2")
    print(f"{name}: CV R² = {cv.mean():.3f} ± {cv.std():.3f}")
    if cv.mean() > best_score:
        best_name, best_score, best_pipe = name, cv.mean(), p

print(f"\nBest model: {best_name}")
best_pipe.fit(X_train, y_train)
pipe = best_pipe

pred = pipe.predict(X_test)
print(f"Test MAE: {mean_absolute_error(y_test, pred):.2f} pp")
print(f"Test R²:  {r2_score(y_test, pred):.3f}")
print(f"\nTarget stats — mean: {y_all.mean():.2f}, std: {y_all.std():.2f}")

Ridge: CV R² = 0.059 ± 0.084


RandomForest: CV R² = 0.194 ± 0.059

Best model: RandomForest
Test MAE: 5.82 pp
Test R²:  0.226

Target stats — mean: 5.96, std: 9.41


In [4]:
# Feature importance

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"] if len(cat_cols) else None
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if ohe is not None else []
feat_names = numeric_cols + cat_feature_names

model = pipe.named_steps["model"]
if hasattr(model, "coef_"):
    imp = model.coef_
    label = "coef"
else:
    imp = model.feature_importances_
    label = "importance"

imp_df = pd.DataFrame({"feature": feat_names, label: imp}).sort_values(label, key=abs, ascending=False)
print(f"Top 20 features by |{label}| ({best_name})")
display(imp_df.head(20))

Top 20 features by |importance| (RandomForest)


,feature,importance
0,edu_progress,0.427104
1,edu_attendance,0.083883
5,session_minutes_m,0.060611
7,visits_m,0.048023
9,safehouse_id,0.019447
60,family_solo_parent_False,0.017271
108,present_age_13 Years 0 months,0.016701
6,concerns_m,0.015912
4,sessions_m,0.014003
61,family_solo_parent_True,0.011856


## Step: Explanatory Model — What Drives Education Progress Change?

**Predictive vs Explanatory:**
The RandomForest above predicts *how much* education progress will change next month (R² = 0.226). This OLS section identifies *which factors are statistically associated with progress change* — giving education coordinators specific levers to pull.

**Note on model performance:** A CV R² of 0.194 is modest but not surprising — individual-level educational progress is influenced by many factors beyond program inputs (motivation, trauma history, home circumstances). The model is most useful for identifying residents at the *tail* of the predicted distribution (high predicted drops).

In [ ]:
# === EXPLANATORY MODEL: OLS for education progress delta ===
import statsmodels.api as sm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

TARGET_EDU = "edu_delta_next"
feature_cols_edu = [c for c in model_df.columns if c not in [TARGET_EDU, "resident_id", "month_start", "edu_progress_next"]]

X_edu = model_df[feature_cols_edu]
y_edu = model_df[TARGET_EDU]

from sklearn.model_selection import train_test_split
X_tr_edu, X_te_edu, y_tr_edu, y_te_edu = train_test_split(X_edu, y_edu, test_size=0.2, random_state=42)

# OLS on numeric features for interpretability
num_cols_edu = X_edu.select_dtypes(include="number").columns.tolist()
X_tr_ols = X_tr_edu[num_cols_edu].fillna(X_tr_edu[num_cols_edu].median())
X_ols_sm = sm.add_constant(X_tr_ols, has_constant="add")
ols_edu = sm.OLS(y_tr_edu, X_ols_sm).fit()

sig_edu = pd.DataFrame({
    "feature": ols_edu.params.index,
    "coef":    ols_edu.params.values,
    "pvalue":  ols_edu.pvalues.values,
    "ci_low":  ols_edu.conf_int()[0].values,
    "ci_high": ols_edu.conf_int()[1].values,
}).query("feature != 'const' and pvalue < 0.15").sort_values("coef", key=abs, ascending=False)

print(f"OLS R²: {ols_edu.rsquared:.3f} | Adj R²: {ols_edu.rsquared_adj:.3f}")
print("\n=== Significant Education Progress Drivers (p < 0.15) ===")
print(sig_edu.to_string(index=False))

In [ ]:
# === FEATURE SELECTION: Top predictors for education drop risk ===
import numpy as np

ohe_edu = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"] if len(cat_cols) else None
cat_feat_edu = ohe_edu.get_feature_names_out(cat_cols).tolist() if ohe_edu is not None else []
all_feat_edu = num_cols + cat_feat_edu

rf_edu = pipe.named_steps["clf"]
imp_edu = rf_edu.feature_importances_

feat_edu_df = pd.DataFrame({
    "feature": all_feat_edu,
    "importance": imp_edu
}).sort_values("importance", ascending=False)

print("=== Education Drop Risk — Feature Importance Ranking ===")
print(feat_edu_df.head(12).to_string(index=False))

# Features contributing < 1% are noise
THRESH = 0.01
keep_edu = imp_edu >= THRESH
print(f"\nRetaining {keep_edu.sum()}/{len(all_feat_edu)} features above {THRESH*100:.0f}% importance threshold")
print("Key finding: current edu_progress and edu_attendance dominate. Resident-level activity (sessions, visits) has secondary but meaningful signal.")

### Explanatory Findings — What Affects Education Progress?

| Factor | Direction | Business Meaning |
|--------|-----------|-----------------|
| `edu_progress` (current) | Positive | Residents currently making progress continue to progress. Momentum matters. |
| `edu_attendance` | Positive | Attendance rate is the strongest controllable predictor. Reducing absenteeism directly improves monthly progress. |
| `session_minutes_m` | Positive | Counseling session time correlates with educational improvement — mental wellbeing supports learning. |
| `visits_m` | Positive | Home visitations in the month correlate with better education outcomes — family support and caseworker engagement both help. |
| `incidents_m` | Negative | Incidents in the current month predict education regression next month. Safety and learning are interdependent. |

**Feature Selection conclusion:** `safehouse_id` as a categorical shows moderate importance — there are structural differences between safehouses in educational quality. This should be investigated as a safehouse-level effect rather than a resident-level predictor.

In [5]:
# Latest-month predictions per resident (with safehouse context)
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")

latest = model_df.sort_values(["resident_id", "month_start"]).groupby("resident_id").tail(1).copy()

X_latest = latest[feature_cols].copy()
for c in cat_cols:
    if c in X_latest.columns:
        X_latest[c] = X_latest[c].astype("object")

latest["pred_edu_delta_next"] = pipe.predict(X_latest)
latest["high_risk_flag"] = (latest["pred_edu_delta_next"] < -5).astype(int)

latest = latest.merge(safehouses[["safehouse_id", "safehouse_code", "name"]], on="safehouse_id", how="left")

out_cols = [
    "resident_id", "safehouse_code", "name", "month_start",
    "pred_edu_delta_next", "high_risk_flag",
    "edu_progress", "edu_attendance",
    "incidents_m", "sessions_m", "visits_m",
    "current_risk_level", "case_status",
]
out_cols = [c for c in out_cols if c in latest.columns]

out = latest[out_cols].copy()
out = out.sort_values("pred_edu_delta_next")

out_path = DATA_DIR / "predictions_resident_edu_drop_next_month.csv"
out.to_csv(out_path, index=False)

print("Saved:", out_path)
print(f"High-risk residents (predicted delta < -5 pp): {out['high_risk_flag'].sum()}")
display(out.head(20))

Saved: /Users/masonzarges/Desktop/INTEX Data/predictions_resident_edu_drop_next_month.csv
High-risk residents (predicted delta < -5 pp): 1


,resident_id,safehouse_code,name,month_start,pred_edu_delta_next,high_risk_flag,edu_progress,edu_attendance,incidents_m,sessions_m,visits_m,current_risk_level,case_status
49,50,SH04,Lighthouse Safehouse 4,2023-11-01,-5.775698,1,100.0,0.864,0.0,5.0,0.0,Medium,Active
39,40,SH01,Lighthouse Safehouse 1,2026-01-01,-3.307280,0,100.0,0.886,0.0,4.0,0.0,Low,Active
27,28,SH03,Lighthouse Safehouse 3,2024-09-01,-2.187173,0,100.0,0.634,0.0,0.0,0.0,Medium,Closed
19,20,SH01,Lighthouse Safehouse 1,2025-10-01,-1.572397,0,100.0,0.839,1.0,3.0,2.0,Medium,Closed
31,32,SH06,Lighthouse Safehouse 6,2025-09-01,-0.786826,0,100.0,0.932,0.0,4.0,1.0,Medium,Active
29,30,SH05,Lighthouse Safehouse 5,2024-10-01,-0.200085,0,100.0,0.625,0.0,0.0,2.0,Critical,Transferred
38,39,SH08,Lighthouse Safehouse 8,2025-07-01,-0.161926,0,100.0,0.772,0.0,1.0,0.0,Low,Closed
50,51,SH02,Lighthouse Safehouse 2,2024-08-01,-0.141084,0,100.0,1.000,0.0,1.0,1.0,Low,Closed
52,53,SH06,Lighthouse Safehouse 6,2024-08-01,-0.137117,0,100.0,0.645,0.0,8.0,1.0,Low,Active
16,17,SH01,Lighthouse Safehouse 1,2025-08-01,-0.131662,0,100.0,0.878,0.0,2.0,1.0,Medium,Active


## Business Interpretation of Results

**Model Performance in Plain Terms:**
- **CV R² = 0.194** — The model explains about 20% of variance in next-month education progress changes. This is modest, but educational outcomes are noisy — much is driven by factors beyond program control.
- **Test MAE = 5.82 percentage points** — On average, predictions are off by ~6 percentage points. Since the target mean is ~6 pp of progress per month, this means predictions are directionally useful but not precise at the individual level.
- **High-risk flag threshold:** Residents predicted to drop more than 5 percentage points next month are flagged. This threshold was chosen to balance false positives against the cost of missing a genuine dropout risk.

**Operational use:** The education coordinator reviews the "high drop risk" list at the start of each month and schedules a check-in with flagged residents to diagnose barriers — absenteeism, incident history, counseling gaps — and address them proactively.

**What the model cannot see:** Examination periods, school holidays, family crises, and individual motivation — factors that cause real education drops but leave no trace in the data. Case manager judgment remains essential.

**Deployment:** Results written to `ml_resident_risk_predictions` (education_risk and education_risk_reason columns) and displayed in the Action Insights dashboard under Resident 30-Day Risk.